### Merge 7-files CVS & 2-files json

In [0]:
%sql
CREATE  OR REPLACE TABLE restaurant_project.silver.restaurant_final AS
SELECT * FROM restaurant_project.bronze.restaurant_1
UNION ALL
SELECT * FROM restaurant_project.bronze.restaurant_2
UNION ALL
SELECT * FROM restaurant_project.bronze.restaurant_3
UNION ALL
SELECT * FROM restaurant_project.bronze.restaurant_4
UNION ALL
SELECT * FROM restaurant_project.bronze.restaurant_5
UNION ALL
SELECT * FROM restaurant_project.bronze.restaurant_6
UNION ALL
SELECT * FROM restaurant_project.bronze.restaurant_7
UNION ALL
SELECT * FROM restaurant_project.bronze.restaurant_json_1
UNION ALL
SELECT * FROM restaurant_project.bronze.restaurant_json_2

num_affected_rows,num_inserted_rows


In [0]:
%sql
select count(*) from restaurant_project.silver.restaurant_final

count(*)
11110000


### import Table as DataFrame to clean it using PySpark

In [0]:
%python
df = spark.read.table("restaurant_project.silver.restaurant_final")

#### Check for NUlls :

In [0]:
%python
from pyspark.sql.functions import col, count, when

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+-----+----------------+--------+----------+----+--------+---------+-----+--------+--------+------------+------+--------------+----------+-----------+------+----------+
|_line|_fivetran_synced|order_id|order_date|hour|category|item_name|price|quantity|discount|total_amount|branch|payment_method|order_type|customer_id|rating|is_weekend|
+-----+----------------+--------+----------+----+--------+---------+-----+--------+--------+------------+------+--------------+----------+-----------+------+----------+
|    0|               0|       0|         0|   0|       0|        0|    0|       0|       0|           0|     0|             0|         0|          0|     0|         0|
+-----+----------------+--------+----------+----+--------+---------+-----+--------+--------+------------+------+--------------+----------+-----------+------+----------+



In [0]:
df=df.drop("_line", "_fivetran_synced")

In [0]:
df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- hour: long (nullable = true)
 |-- category: string (nullable = true)
 |-- item_name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: long (nullable = true)
 |-- discount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- branch: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_type: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- rating: long (nullable = true)
 |-- is_weekend: long (nullable = true)



#### Check for Duplicates

In [0]:
%python
from pyspark.sql.functions import count

df.groupBy(df.columns).count().filter("count > 1").show()

+--------+----------+----+--------+---------+-----+--------+--------+------------+------+--------------+----------+-----------+------+----------+-----+
|order_id|order_date|hour|category|item_name|price|quantity|discount|total_amount|branch|payment_method|order_type|customer_id|rating|is_weekend|count|
+--------+----------+----+--------+---------+-----+--------+--------+------------+------+--------------+----------+-----------+------+----------+-----+
+--------+----------+----+--------+---------+-----+--------+--------+------------+------+--------------+----------+-----------+------+----------+-----+



In [0]:
%python
df.dropDuplicates()

DataFrame[order_id: bigint, order_date: date, hour: bigint, category: string, item_name: string, price: double, quantity: bigint, discount: double, total_amount: double, branch: string, payment_method: string, order_type: string, customer_id: bigint, rating: bigint, is_weekend: bigint]

In [0]:
%python
df=df.filter(df["price"] >= 0)

In [0]:
%python
df.count()

11058656

In [0]:
df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- hour: long (nullable = true)
 |-- category: string (nullable = true)
 |-- item_name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: long (nullable = true)
 |-- discount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- branch: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_type: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- rating: long (nullable = true)
 |-- is_weekend: long (nullable = true)



In [0]:
df.write.format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("restaurant_project.silver.restaurant_final")

In [0]:
%sql
select * from restaurant_project.silver.restaurant_final limit 10;

order_id,order_date,hour,category,item_name,price,quantity,discount,total_amount,branch,payment_method,order_type,customer_id,rating,is_weekend
922676,2025-04-13,23,مشويات,كباب,142.23,5,0.1,640.02,الإسكندرية,Cash,Dine-in,88724,3,1
930611,2022-03-17,12,محاشي,محشي باذنجان,64.94,3,0.0,194.83,الإسكندرية,Cash,Takeaway,114319,4,0
938546,2020-05-29,16,مشروبات,عصير قصب,20.4,4,0.1,73.43,الإسكندرية,Cash,Takeaway,10373,3,1
946481,2021-11-19,19,محاشي,محشي ورق عنب,96.13,3,0.0,288.4,طنطا,Cash,Dine-in,51427,4,1
956472,2024-08-07,18,طواجن,طاجن بامية,146.09,3,0.0,438.26,القاهرة,Card,Dine-in,178248,5,0
964407,2025-06-08,10,مشويات,شيش طاووق,161.23,2,0.0,322.47,القاهرة,Wallet,Takeaway,54609,4,1
972342,2022-10-26,15,مقبلات,طحينة,62.8,3,0.0,188.39,طنطا,Wallet,Delivery,156,4,0
980277,2023-02-12,22,مشويات,كباب,150.56,5,0.1,677.52,الإسكندرية,Cash,Delivery,95926,4,1
657513,2024-03-16,17,مقبلات,بابا غنوج,51.21,3,0.0,153.63,القاهرة,Wallet,Delivery,111781,5,1
665448,2022-06-26,19,مقبلات,بابا غنوج,43.37,4,0.1,156.14,الإسكندرية,Cash,Dine-in,109173,4,1


### Create Date Dimension

In [0]:
%sql
CREATE OR REPLACE TABLE restaurant_project.gold.dim_date
USING DELTA AS
SELECT DISTINCT
    concat(
        date_format(order_date, 'yyyyMMdd'),
        lpad(hour, 2, '0') )  AS Date_PK,
    order_date,
    day(order_date) AS day,
    month(order_date) AS month,
    year(order_date) AS year,
    dayofweek(order_date) AS day_of_week,
    is_weekend,
    hour
FROM restaurant_project.silver.restaurant_final;

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from restaurant_project.gold.dim_date limit 10;

Date_PK,order_date,day,month,year,day_of_week,is_weekend,hour
2024071016,2024-07-10,10,7,2024,4,0,16
2022011121,2022-01-11,11,1,2022,3,0,21
2025082616,2025-08-26,26,8,2025,3,0,16
2020110916,2020-11-09,9,11,2020,2,0,16
2021071916,2021-07-19,19,7,2021,2,0,16
2025090210,2025-09-02,2,9,2025,3,0,10
2024052520,2024-05-25,25,5,2024,7,1,20
2025020411,2025-02-04,4,2,2025,3,0,11
2021091814,2021-09-18,18,9,2021,7,1,14
2024011917,2024-01-19,19,1,2024,6,1,17


### Create item_Dim

In [0]:
%sql
CREATE or REPLACE TABLE restaurant_project.gold.dim_product
USING DELTA AS
SELECT 
    monotonically_increasing_id() AS item_pk,
    * 
    from (select DISTINCT 
            item_name,
            category
          FROM restaurant_project.silver.restaurant_final);

num_affected_rows,num_inserted_rows


In [0]:
%sql
select count(distinct *)from restaurant_project.gold.dim_product

count(DISTINCT*)
15


### Create Customer_Dim

In [0]:
%sql
CREATE or REPLACE TABLE restaurant_project.gold.dim_customer
USING DELTA AS
SELECT 
    monotonically_increasing_id() AS customer_pk,
    * 
    from (select DISTINCT 
          customer_id
          FROM restaurant_project.silver.restaurant_final);

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from restaurant_project.gold.dim_customer limit 10;

customer_pk,customer_id
0,122751
1,72819
2,83992
3,173902
4,117212
5,176841
6,3298
7,112725
8,92404
9,92243


In [0]:
%sql
select count(*) from restaurant_project.gold.dim_customer 

count(*)
199999


### Create Branch_Dim

In [0]:
%sql
CREATE or REPLACE TABLE restaurant_project.gold.dim_brach
USING DELTA AS
SELECT 
    monotonically_increasing_id() AS branch_pk,
    * 
    from (select DISTINCT 
          branch
          FROM restaurant_project.silver.restaurant_final);

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from restaurant_project.gold.dim_brach ;

branch_pk,branch
0,القاهرة
1,الجيزة
2,أسيوط
3,طنطا
4,المنصورة
5,الإسكندرية


### Create Payment_Method Dim

In [0]:
%sql
CREATE or REPLACE TABLE restaurant_project.gold.dim_payment
USING DELTA AS
SELECT 
    monotonically_increasing_id() AS payment_pk,
    * 
    from (select DISTINCT 
          payment_method
          FROM restaurant_project.silver.restaurant_final);

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from restaurant_project.gold.dim_payment 

payment_pk,payment_method
0,Wallet
1,Cash
2,Card


### Create Order_Type Dim

In [0]:
%sql
CREATE or REPLACE TABLE restaurant_project.gold.dim_order_type
USING DELTA AS
SELECT 
    monotonically_increasing_id() AS order_type_pk,
    * 
    from (select DISTINCT 
          order_type
          FROM restaurant_project.silver.restaurant_final);

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from restaurant_project.gold.dim_order_type

order_type_pk,order_type
0,Delivery
1,Dine-in
2,Takeaway


### Create Fact_Order

In [0]:
%sql
CREATE OR REPLACE TABLE restaurant_project.gold.fact_orders 
USING DELTA AS
SELECT
    o.order_id ,
    d.Date_PK as date_FK,
    p.item_pk as product_FK,
    c.customer_pk as customer_FK,
    b.branch_pk as branch_FK,
    pay.payment_pk as payment_FK,
    ot.order_type_pk as order_type_FK,

    -- Measures
    o.price,
    o.quantity,
    o.discount,
    o.total_amount,
    o.rating

FROM restaurant_project.silver.restaurant_final o

LEFT JOIN restaurant_project.gold.dim_date  d
    ON o.order_date = d.order_date
    AND o.hour = d.hour

LEFT JOIN restaurant_project.gold.dim_product p
    ON o.item_name = p.item_name
    AND o.category = p.category
    
LEFT JOIN restaurant_project.gold.dim_customer c
    ON o.customer_id = c.customer_id

LEFT JOIN restaurant_project.gold.dim_brach b
    ON o.branch = b.branch

LEFT JOIN restaurant_project.gold.dim_payment pay
    ON o.payment_method = pay.payment_method

LEFT JOIN restaurant_project.gold.dim_order_type ot
    ON o.order_type = ot.order_type;

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from restaurant_project.gold.fact_orders limit 10;

order_id,date_FK,product_FK,customer_FK,branch_FK,payment_FK,order_type_FK,price,quantity,discount,total_amount,rating
1,2020061316,8,131219,0,1,2,39.16,4,0.1,140.96,4
2,2021112917,7,189658,4,2,2,53.95,2,0.0,107.9,4
3,2025030713,3,154344,0,0,2,120.81,5,0.1,543.65,5
4,2024102311,4,68194,1,1,0,33.69,1,0.0,33.69,4
5,2021013114,11,123857,0,2,0,108.99,4,0.1,392.37,4
6,2023022117,9,166272,3,1,1,144.16,2,0.0,288.32,4
7,2022052717,2,47949,1,2,0,98.25,3,0.0,294.76,3
8,2020033111,2,61733,0,1,1,115.43,2,0.0,230.87,4
9,2021082320,6,147819,0,2,0,129.52,2,0.0,259.05,3
10,2020070922,8,184926,1,1,0,56.64,3,0.0,169.91,3


In [0]:
%sql
select count(*) from restaurant_project.gold.fact_orders

count(*)
11058656
